# Workshop 3 · 01 — Stammdaten heilen (Aufgabe)

**Ziel:** Unvollständige und uneinheitliche Stammdaten sowie Freitextmeldungen in **saubere, strukturierte Spalten** verwandeln — mit **AI Functions**.

**Deine Aufgabe ist nur der AI-Aufruf.** Label-Listen, JSON-Schema, Parsing und Speichern sind bereits vorgegeben — du setzt an den markierten `# TODO`-Stellen den korrekten `ai.*`-Aufruf ein.

1. `ai.classify` — `depot` und `baureihe` auf einheitliche Werte mappen.
2. `ai.generate_response` — fehlende Felder per JSON-Schema aus dem Freitext ziehen.
3. Ergebnis als Tabelle `asset_clean` (vorgegeben).

Referenz: https://learn.microsoft.com/fabric/data-science/ai-functions/overview

## Schritt 0 — Daten ansehen
Ausführen: uneinheitliche/leere Stammdaten, die Information steckt nur im Freitext.

In [ ]:
import synapse.ml.spark.aifunc as aifunc  # registriert den .ai Accessor auf Spark-DataFrames

df = spark.table("asset_meldungen")
display(df.select("meldung_id", "baureihe", "depot", "komponente", "freitext_meldung"))

## Schritt 1 — Stammdaten normalisieren mit `ai.classify`

Die Ziel-Labels sind vorgegeben. **Aufgabe:** Wende `ai.classify` an und erzeuge die Spalten `depot_norm` und `baureihe_norm`.

Signatur: `df.ai.classify(input_col=..., output_col=..., labels=...)` — die Aufrufe lassen sich verketten.

In [ ]:
# This code uses AI. Always review output for mistakes.

# Vorgegeben: einheitliche Ziel-Labels
depot_labels = ["Hamburg-Eidelstedt", "Frankfurt-Griesheim", "Muenchen-Pasing",
                "Leipzig", "Berlin-Rummelsburg"]
baureihe_labels = ["ICE4", "ICE3", "IC2", "Talent2", "Flirt3", "Doppelstock", "unbekannt"]

# TODO: ai.classify anwenden -> Spalten "depot_norm" und "baureihe_norm"
#       z.B. df.ai.classify(input_col="depot", output_col="depot_norm", labels=depot_labels)
#            .ai.classify(input_col="baureihe", output_col="baureihe_norm", labels=baureihe_labels)
norm = ...   # <-- hier die verketteten ai.classify-Aufrufe einsetzen

display(norm.select("meldung_id", "depot", "depot_norm", "baureihe", "baureihe_norm"))

## Schritt 2 — fehlende Felder aus dem Freitext ziehen mit `ai.generate_response`

JSON-Schema und Prompt sind vorgegeben. **Aufgabe:** Wende `ai.generate_response` auf `norm` an und schreibe das Ergebnis in die Spalte `extract_json`.

Signatur: `norm.ai.generate_response(prompt=prompt, is_prompt_template=True, response_format=meldung_schema, output_col="extract_json")`

In [ ]:
# This code uses AI. Always review output for mistakes.

# Vorgegeben: JSON-Schema fuer die zu extrahierenden Felder
meldung_schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "meldung",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "komponente": {"type": "string"},
                "ort_am_fahrzeug": {"type": "string"},
                "schweregrad": {"type": "integer", "description": "1 kosmetisch bis 5 sicherheitskritisch"},
                "sicherheitsrelevant": {"type": "boolean"},
                "prioritaet": {"type": "string", "enum": ["hoch", "mittel", "niedrig"]},
            },
            "required": ["komponente", "ort_am_fahrzeug", "schweregrad", "sicherheitsrelevant", "prioritaet"],
            "additionalProperties": False,
        },
    },
}

# Vorgegeben: Prompt-Template (referenziert die Freitext-Spalte)
prompt = ("Extrahiere aus dieser Werkstatt-Meldung strukturierte Felder: {freitext_meldung}. "
          "Schweregrad 1 (kosmetisch) bis 5 (sicherheitskritisch).")

# TODO: ai.generate_response anwenden -> Spalte "extract_json"
extracted = ...   # <-- hier den ai.generate_response-Aufruf einsetzen

display(extracted.select("meldung_id", "freitext_meldung", "extract_json"))

## Schritt 3 — JSON parsen und speichern (vorgegeben)
Kein AI-Schritt — parst `extract_json` in typisierte Spalten und speichert die Tabelle `asset_clean`. Einfach ausführen.

In [ ]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType

ext_schema = StructType([
    StructField("komponente", StringType()),
    StructField("ort_am_fahrzeug", StringType()),
    StructField("schweregrad", IntegerType()),
    StructField("sicherheitsrelevant", BooleanType()),
    StructField("prioritaet", StringType()),
])

clean = (extracted
         .withColumn("e", from_json(col("extract_json"), ext_schema))
         .select("meldung_id", "wagon_id", "baureihe_norm", "depot_norm",
                 col("e.komponente").alias("komponente"),
                 col("e.ort_am_fahrzeug").alias("ort"),
                 col("e.schweregrad").alias("schweregrad"),
                 col("e.sicherheitsrelevant").alias("sicherheitsrelevant"),
                 col("e.prioritaet").alias("prioritaet"),
                 "freitext_meldung", "kunden_kommentar", "foto_pfad"))

clean.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("asset_clean")
display(clean.orderBy(col("schweregrad").desc()))

### Ergebnis-Check
Tabelle `asset_clean` mit normalisierten Stammdaten und aus Freitext extrahierten Feldern. Prüfe: höchster `schweregrad` oben, keine leeren `depot_norm`/`komponente`.